Problem Statement

---



<h6>This project aims to identify high-risk patients based on healthcare data to support better decision-making and resource allocation.</h6>

Import Libraries

---



In [378]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

Load Dataset

---



In [379]:
df = pd.read_csv("healthcare_dataset.csv")
df.head()

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


Any Missing Values

---



In [380]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55500 entries, 0 to 55499
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                55500 non-null  object 
 1   Age                 55500 non-null  int64  
 2   Gender              55500 non-null  object 
 3   Blood Type          55500 non-null  object 
 4   Medical Condition   55500 non-null  object 
 5   Date of Admission   55500 non-null  object 
 6   Doctor              55500 non-null  object 
 7   Hospital            55500 non-null  object 
 8   Insurance Provider  55500 non-null  object 
 9   Billing Amount      55500 non-null  float64
 10  Room Number         55500 non-null  int64  
 11  Admission Type      55500 non-null  object 
 12  Discharge Date      55500 non-null  object 
 13  Medication          55500 non-null  object 
 14  Test Results        55500 non-null  object 
dtypes: float64(1), int64(2), object(12)
memory usage: 6.4

Unique Medical Conditions

---



In [381]:
medical = df['Medical Condition'].unique()
print(medical)

['Cancer' 'Obesity' 'Diabetes' 'Asthma' 'Hypertension' 'Arthritis']


KPI's

---



Total Patients

In [382]:
total_patients = len(df)
print("Total Patients :", total_patients)

Total Patients : 55500


Average Billing Amount


In [383]:
Avg_Bill = df['Billing Amount'].mean()
print("Average Billing :",round(Avg_Bill,2))

Average Billing : 25539.32


Average Stay



In [384]:
df['Date of Admission'] = pd.to_datetime(df['Date of Admission'])
df['Discharge Date'] = pd.to_datetime(df['Discharge Date'])

df['Days Stayed'] = (df['Discharge Date'] - df['Date of Admission']).dt.days

In [385]:
avg_stay = df['Days Stayed'].mean()
print("Average Days Stayed :", round(avg_stay,2))

Average Days Stayed : 15.51


High Risk Percentage

In [386]:
df['High_Risk'] = ((df['Days Stayed'] > 7) &(df['Billing Amount'] > df['Billing Amount'].median())).astype(int)

high_risk_pct = df['High_Risk'].mean() * 100
print("High Risk % :", round(high_risk_pct,2))

High Risk % : 38.14


Patient Demographics

---



In [387]:
fig = px.histogram(df, x='Gender', color='Gender', title='Patient Demographics')
fig.update_layout(xaxis_title='Gender', yaxis_title='Number of Patients')
fig.show()

Average Billing Per Age Group

---



In [388]:
df['Age_Group'] = pd.cut(df['Age'], bins=[0,18,40,65,100],labels=['Child','Young Adult','Adult','Senior'])
Avg_Bill_Age = df.groupby('Age_Group',observed=False)['Billing Amount'].mean().reset_index()

fig = px.bar(Avg_Bill_Age, x='Age_Group', y='Billing Amount', title='Average Billing Per Age Group', color='Age_Group')
fig.update_layout(xaxis_title='Age Group', yaxis_title='Average Billing Amount', legend_title='Age Group')
fig.show()

<p>Average billing is fairly consistent across age groups, with children showing slightly higher costs, while other age groups have similar billing patterns.</p>

Frequency of Medical Conditions in Hospital

---



In [389]:
fig = px.histogram(df, x='Medical Condition', color='Medical Condition', title='Frequency of Medical Conditions in Hospital')
fig.update_layout(xaxis_title='Condition Type', yaxis_title='Number of Patients')
fig.show()

<p>Medical conditions are almost evenly distributed, with only slight differences (Arthritis 9308, Diabetes 9304 vs others are approximately 9231). This indicates no condition significantly dominates patient volume.<p>

Cost Analysis by Condition

---



In [390]:
condition_costs = df.groupby('Medical Condition')['Billing Amount'].mean().sort_values(ascending=False).reset_index()
fig = px.bar(condition_costs, x='Medical Condition', y='Billing Amount', title='Cost Analysis by Condition', color='Medical Condition')
fig.update_layout(xaxis_title='Medical Condition', yaxis_title='Average Billing Amount')
fig.show()

<p>Obesity has the highest average billing among all conditions, indicating it is the most costly condition and may require more intensive treatment and resources.</p>

Length of Stay

---



In [391]:
fig =px.box(df, x='Medical Condition', y='Days Stayed', color='Medical Condition', title='Length of Stay by Condition')
fig.show()

<p>Hospital stay duration is largely consistent across all conditions, with only a slight increase for Asthma patients. No condition shows significantly longer or more variable stays.</p>

Stay Duration vs Total Billing

---



In [392]:
top = df.groupby('Medical Condition')['Billing Amount'].mean().idxmax()

df_trend = df[df['Medical Condition'] == top].groupby('Days Stayed')['Billing Amount'].mean().reset_index()

px.line(df_trend, x='Days Stayed', y='Billing Amount',
        title=f'Stay Duration vs Billing by Top Condition : {top}').show()

<p>For Obesity patients, billing is generally high and slightly increases with more days stayed, but there are some ups and downs, showing that cost is not only based on stay duration.
</p>

Insurance vs Cost

---





In [393]:
Insurance_Analysis = df.groupby('Insurance Provider')['Billing Amount'].mean().sort_values(ascending=False).reset_index()

fig = px.bar(Insurance_Analysis, x='Insurance Provider', y='Billing Amount', title='Insurance vs Cost', color='Insurance Provider')
fig.update_layout(xaxis_title='Insurance Type', yaxis_title='Average Billing Amount')
fig.show()

<p>The average billing amount is almost the same across all insurance providers, showing that cost does not vary much based on insurance type.</p>

High Risk Patients

---



In [394]:
df['Risk_Label'] = df['High_Risk'].map({0: 'Low Risk', 1: 'High Risk'})
px.pie(df,
       names='Risk_Label',
       color='Risk_Label',
       color_discrete_map={'Low Risk': 'green', 'High Risk': 'red'},
       title='High Risk vs Low Risk Percentage').show()

In [395]:
fig = px.histogram(df, x='Medical Condition', color='Risk_Label',barmode='group',color_discrete_map={'Low Risk':'green', 'High Risk':'red'},
                   title='High Risk Patients Count by Medical Condition')
fig.update_layout(legend_title_text='Risk Level')

fig.show()

<p>High-risk cases are not dominated by a single medical condition, but Obesity and Arthritis contribute slightly more to the high-risk population, suggesting higher resource utilization in these categories.</p>

Top Cost Condition and Insurance

---



In [396]:
top_condition = df.groupby('Medical Condition')['Billing Amount'].mean().idxmax()

top_insurance = df.groupby('Insurance Provider')['Billing Amount'].mean().idxmax()

print("Highest Cost Condition:", top_condition)
print("Highest Cost Insurance:", top_insurance)

Highest Cost Condition: Obesity
Highest Cost Insurance: Medicare


Models

---




In [397]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df_model = df.copy()

for col in df_model.select_dtypes(include='object'):
    df_model[col] = LabelEncoder().fit_transform(df_model[col])

Train Model

---




In [398]:
X = df_model[['Age','Gender','Blood Type','Medical Condition','Insurance Provider','Billing Amount']]
y = df_model['High_Risk']

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Logistic Regression

---



In [399]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")

Logistic Regression Accuracy: 81.42%


Random Forest Classifier

---



In [400]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Random Forest Classifier Accuracy: {accuracy*100:.2f}%")

Random Forest Classifier Accuracy: 88.18%


Decision Tree

---



In [401]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=8)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")

Decision Tree Accuracy: 88.30%
